# 🏛️ Итоговое домашнее задание
# «Аналитик реестра — Обработка данных ЕФРСБ»





Вы — юрист, анализирующий данные из **Единого федерального реестра сведений о банкротстве (ЕФРСБ)**.
Вам поступила выгрузка в формате JSON с информацией о сообщениях по делам о банкротстве за 2025 год.

**Ваша задача** — извлечь ключевые данные, очистить от мусора, сопоставить с реестром организаций
и сформировать аналитический отчёт для руководства.

---
### 📋 Как выполнять и сдавать задание

**Где выполнять:**
- **Вариант 1 (рекомендуется):** Загрузите файл `.ipynb` в [Google Colab](https://colab.research.google.com/), загрузите папку `final_hw_data/` в среду выполнения (например, через боковую панель «Файлы» или смонтировав Google Drive).
- **Вариант 2:** Работайте локально в Jupyter Notebook / JupyterLab. Убедитесь, что папка `final_hw_data/` находится в той же директории, что и файл задания.

**Как сдавать на проверку:**
- Отправьте **ссылку на Google Colab** с общим доступом («Любой, у кого есть ссылка — комментатор»), **ИЛИ**
- Отправьте **ссылку на GitHub-репозиторий**, в котором лежит выполненный `.ipynb`-файл и папка `final_hw_data/` с результатами.

> ⚠️ Убедитесь, что ссылка открывается в режиме просмотра и все ячейки выполнены (результаты видны).

---

### 📦 Исходные данные

В папке `final_hw_data/` находятся 3 файла:

| Файл | Описание |
|------|----------|
| `bankruptcy_messages.json` | Сообщения ЕФРСБ |
| `organizations.json` | Реестр организаций |
| `priority_cases.txt` | Номера приоритетных дел |



---
# 🟢 Загрузка и кросс-референс (0-2 балла)
---

### Задание 1.1 — Загрузка данных из файлов

Загрузите данные из трёх файлов:
- `bankruptcy_messages.json` → список `messages`
- `organizations.json` → список `organizations`
- `priority_cases.txt` → список `priority_case_numbers`

Выведите количество загруженных записей из каждого источника.

In [13]:
import json

with open('bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    banc_list = json.load(f)
count_b = len(banc_list)
print(f'В файле "bankruptcy_messages.json" записей: {count_b}')

with open('organizations.json', 'r', encoding='utf-8') as f:
    organization_list = json.load(f)
    count_o = len(organization_list)
print(f'\nВ файле "organizations.json" записей: {count_o}')

with open('priority_cases.txt', 'r', encoding='utf-8') as f:
    count_p = sum (1 for line in f if line.strip())
print(f'\nВ файле "priority_cases.txt" записей: {count_p}')


В файле "bankruptcy_messages.json" записей: 54

В файле "organizations.json" записей: 30

В файле "priority_cases.txt" записей: 10


### Задание 1.2 — Словарь организаций

Создайте словарь `org_by_inn`, где ключ — ИНН (строка), а значение — словарь с данными об организации.
Используйте **dict comprehension**.

Выведите количество организаций в словаре и информацию по ИНН `"7701234567"`.

In [16]:
import json

with open('organizations.json', 'r', encoding='utf-8') as f:
    org_list = json.load(f)

org_by_inn = {org['inn']: org for org in org_list}

# org_by_inn = {}

# for org in org_list:
#     inn = org['inn']
#     org_by_inn[inn] = org

count_inn = len(org_by_inn)


print(f'\nВ файле " org_by_inn" записей: {count_inn}.\n')
print("=" * 42)
print(f'Сведения об организации: {org_by_inn['7701234567']}')



В файле " org_by_inn" записей: 30.

Сведения об организации: {'inn': '7701234567', 'ogrn': '1027700123456', 'name': 'ООО "Альфа-Строй"', 'address': 'г. Москва, ул. Строителей, д. 15', 'region': 'Москва'}


### Задание 1.3 — Кросс-референс: связываем сообщения с организациями

Для каждого сообщения добавьте поле `org_name` (название организации) и `region`,
используя словарь `org_by_inn` и метод `.get()`.
Если организация не найдена — ставьте значение `None`.

Посчитайте, сколько сообщений удалось связать с организацией,
а сколько — не удалось.

In [ ]:
with open('bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    banc_list = json.load(f)

    for msg in banc_list:
        publisher_inn = msg['publisher_inn']
        org = org_by_inn.get(publisher_inn)
        if org is not None:
            msg["org_name"] = org["name"]
            msg["region"] = org["region"]
        else:
            msg["org_name"] = None
            msg["region"] = None
            
    count_all = len(banc_list) 
    count_none = sum(1 for msg in banc_list if msg["org_name"] is None)

messages = banc_list

print(f"Удалось связать записи: {count_all - count_none} шт.") 
print(f"Не удалось связать записи:  {count_none} шт.")
   

Удалось связать записи: 51 шт.
Не удалось связать записи:  3 шт.


### Задание 1.4 — Множества и приоритетные дела

1. Создайте множество `priority_set` из списка `priority_case_numbers`.
2. Создайте множество `message_case_set` из номеров дел в `messages`
   (используйте list comprehension + filter для непустых номеров).
3. Найдите пересечение — номера дел, которые есть и в приоритетных, и в сообщениях (`&`).
4. Выведите результат.

In [69]:
priority_case_numbers = []
with open('priority_cases.txt', 'r', encoding='utf-8') as f:
    for line in f:
        priority_case_numbers.append(line.strip())
priority_set = set(priority_case_numbers)

message_case_set = {msg["case_number"] for msg in banc_list if msg["case_number"] is not None}

common_cases = priority_set&message_case_set

print(f'Найдено {len(common_cases)} пересечений:')
print(common_cases)


Найдено 10 пересечений:
{'А60-66666/2025', 'А60-22222/2025', 'А60-99999/2025', 'А60-56789/2025', 'А60-11111/2025', 'А60-44444/2025', 'А60-77777/2025', 'А60-33333/2025', 'А60-88888/2025', 'А60-12345/2025'}


---
# 🟡 Очистка и валидация (0-3 балла)
---

### Задание 2.1 — Функция парсинга дат

Напишите функцию `parse_date(date_str)`, которая принимает строку с датой
и возвращает объект `datetime` или `None`, если распарсить не удалось.

Форматы для обработки:
- `"DD.MM.YYYY"` — `strptime`
- `"YYYY-MM-DD"` — `fromisoformat`
- `"DD месяца YYYY г."` — замена русских месяцев + `strptime`
- `"DD/MM/YYYY HH:MM"` — `strptime`

Обрабатывайте `None` и пустые строки через `try/except`.

In [70]:
from datetime import datetime

russian_months = {
    'января': '01', 'февраля': '02', 'марта': '03',
    'апреля': '04', 'мая': '05', 'июня': '06',
    'июля': '07', 'августа': '08', 'сентября': '09',
    'октября': '10', 'ноября': '11', 'декабря': '12'
}


def parse_date(date_str):
    if date_str is None:
        return None
    date_str = date_str.strip()
    if not date_str:
        return None
    try:
        return datetime.strptime(date_str, '%d.%m.%Y')
    except ValueError:
        pass
    try:
        return datetime.fromisoformat(date_str)
    except ValueError:
        pass
    try:
        normalized = date_str
        for month_name, month_num in russian_months.items():
            normalized = normalized.replace(month_name, month_num)
        normalized = normalized.replace(' г.', '')
        return datetime.strptime(normalized, '%d %m %Y')
    except ValueError:
        pass
    try:
        return datetime.strptime(date_str, '%d/%m/%Y %H:%M')
    except ValueError:
        pass  
      
    return None

# print(parse_date('15.01.2024'))            # DD.MM.YYYY
# print(parse_date('2024-01-15'))            # YYYY-MM-DD (fromisoformat)
# print(parse_date('2024-01-15T12:00:00'))   # ISO с временем (fromisoformat)
# print(parse_date('3 февраля 2024 г.'))     # русский месяц
# print(parse_date('15/01/2024 14:30'))      # DD/MM/YYYY HH:MM
# print(parse_date(None))                    # None
# print(parse_date(''))                      # None


### Задание 2.2 — Функция валидации сообщения

Напишите функцию `validate_message(msg)`, которая:

1. Проверяет наличие **обязательных полей**: `publisher_inn`, `msg_text`, `date_published`, `type`, `case_number`.
   Если поле отсутствует — ошибка типа `KeyError`.
2. Проверяет, что **ИНН** состоит из 10 или 12 цифр (метод `.isdigit()`).
3. Парсит дату через `parse_date()`. Если дата не парсится — ошибка.
4. Проверяет, что **тип сообщения** — непустая строка.

Функция возвращает кортеж `(valid_msg, errors)`:
- `valid_msg` — словарь с очищенными данными (или `None`)
- `errors` — список строк с описанием ошибок

In [90]:
filds_reg = ['publisher_inn', 'msg_text', 'date_published', 'type', 'case_number']

def validate_message(msg):
    errors = []

    # 1. Проверка полей
    for field in filds_reg:
        if field not in msg or msg[field] is None or msg[field] == '':
            errors.append(f"Отсутствует поле '{field}'")
        
    if errors:
        return None, errors
    
    # 2.Проверка ИНН
    inn = msg.get('publisher_inn')
    try:
        if not inn.isdigit() or len(inn) not in (10, 12):
            errors.append('ИНН должен состоять из 10 или 12 цифр')
    except (TypeError, AttributeError) as e:
        errors.append(f'Неверный тип ИНН: {e}')

    # 3. Проверка даты
    raw_date = msg.get('date_published')
    parsed_date = parse_date(raw_date)
    if parsed_date is None:
        errors.append('Дата публикации не парсится')

    # 4. Проверка непустой строки
    msg_type = msg.get('type')
    try:
        if not msg_type.strip():
            errors.append('Тип сообщения не должен быть пустой строкой')
    except (TypeError, AttributeError) as e:
        errors.append(f'Тип сообщения: {e}')

    # Вывод ошибок
    if errors:
        return None, errors

    # Вывод очищенного словаря
    valid_msg = {
        'publisher_inn': inn,
        'msg_text': msg.get('msg_text'),
        'date_published': parsed_date,
        'type': msg_type.strip(),
        'case_number': msg.get('case_number'),
        'region': msg.get('region'),
        'org_name': msg.get('org_name'),
    }

    return valid_msg, []

### Задание 2.3 — Массовая валидация

Примените `validate_message()` ко всем сообщениям.
Разделите результат на два списка: `valid_messages` и `error_records`.

Соберите **статистику ошибок** по типам (сколько `KeyError`, `ValueError` и т.д.).

In [91]:
import json

valid_messages = []
error_records = []

for msg in messages:
    valid_msg, errors = validate_message(msg)
    if valid_msg is not None:
        valid_messages.append(valid_msg)
    else:
        error_records.append({
            'publisher_inn': msg.get('publisher_inn', 'UNKNOWN'),
            'case_number': msg.get('case_number', 'UNKNOWN'),
            'errors': errors
        })

print('Статистика:')
print(f'  Всего сообщений: {len(messages)}')
print(f'  ✅ Валидных: {len(valid_messages)}')
print(f'  ❌ С ошибками: {len(error_records)}')

error_stats = {}
for rec in error_records:
    for error in rec['errors']:
        error_stats[error] = error_stats.get(error, 0) + 1

print(f'\n🔍 Распределение ошибок:')
for error_text, count in sorted(error_stats.items(), key=lambda x: x[1], reverse=True):
    print(f'  {count}x — {error_text}')

print(f'\n📋 Детализация:')
for rec in error_records:
    errors_str = '; '.join(rec['errors'])
    print(f' {rec["case_number"]} | ИНН {rec["publisher_inn"]}: {errors_str}')

    


Статистика:
  Всего сообщений: 54
  ✅ Валидных: 48
  ❌ С ошибками: 6

🔍 Распределение ошибок:
  1x — Отсутствует поле 'publisher_inn'
  1x — Отсутствует поле 'date_published'
  1x — Отсутствует поле 'type'
  1x — ИНН должен состоять из 10 или 12 цифр
  1x — Дата публикации не парсится
  1x — Отсутствует поле 'case_number'

📋 Детализация:
 А60-24680/2025 | ИНН : Отсутствует поле 'publisher_inn'
 А60-36925/2025 | ИНН 7701234567: Отсутствует поле 'date_published'
 А60-25814/2025 | ИНН 7702345678: Отсутствует поле 'type'
 А60-12345/2025 | ИНН 7705: ИНН должен состоять из 10 или 12 цифр
 А60-56789/2025 | ИНН 6658123450: Дата публикации не парсится
  | ИНН 7703901234: Отсутствует поле 'case_number'


---
# 🔵 Извлечение данных и аналитика (0-2 балла)
---

### Задание 3.1 — Извлечение сумм из текста

Напишите функцию `extract_amounts(text)`, которая ищет упоминания денежных сумм в тексте сообщения.

Используйте строковые методы.

Ищите по ключевым словам: `"руб."`, `"тыс. руб."`, `"млн руб."`.

Функция возвращает список строк — найденных сумм.

In [104]:
# ищем ключевое слово, 
# обрезаем тест, начиная с ключевого слова, 
# удаляем разделители,
# пробуем преобразовать в инт, 
# часть, которая преобразовалась, забираем, остальное выкидываем

def extract_amounts(text):
    if text is None:
        return []

    amounts = []
    keywords = ['млн руб.', 'тыс. руб.', 'руб.', 'рубль', 'рублей', 'рубля']

    for keyword in keywords:
        start = 0
        while True:
            pos = text.find(keyword, start)
            if pos == -1:
                break

            before = text[:pos].split()

            number_parts = []
            for word in reversed(before):
                clean = word.replace(' ', '')
                try:
                    int(clean)
                    number_parts.append(word)
                except ValueError:
                    break

            if number_parts:
                number_str = ' '.join(reversed(number_parts))
                amounts.append(number_str + ' ' + keyword)

            start = pos + 1

    return amounts[0] if amounts else None

# Проверка
# print(extract_amounts('Долг: 1 000 000 руб. и штраф 25 тыс. руб.'))
# print(extract_amounts('Требования 2 млн руб.'))
# print(extract_amounts('Задолженность 1500000 руб. и штраф 25 тыс. руб.'))
# print(extract_amounts('Требования на сумму 2 млн руб. включены в реестр'))
# print(extract_amounts(None))
# print(extract_amounts('Текст без сумм'))

### Задание 3.2 — Поиск упоминаний арбитражных судов

Напишите функцию `find_court_mentions(text)`, которая проверяет,
содержит ли текст упоминание арбитражного суда.

Ищите подстроку `"АС "` (с пробелом) в тексте через оператор `in`.
Верните `True` / `False`.

In [105]:
def find_court_mentions(text):
    if text is None:
        return False
    return 'АС ' in text


### Задание 3.3 — Извлечение ФИО арбитражных управляющих

Напишите функцию `extract_manager_name(text)`, которая ищет ФИО арбитражного управляющего.

Алгоритм:
1. Проверьте, содержит ли текст подстроку `"арбитражный управляющий"`.
2. Если да — найдите позицию этой подстроки, возьмите текст после неё.
3. С помощью `.split()` извлеките следующие 3 слова (ФИО).
4. Соедините через пробел и верните.
5. Если не найдено — верните `None`.

In [114]:
def extract_manager_name(text):
    if text is None:
        return None
    
    keyword = 'арбитражный управляющий'
    text_lower = text.lower()
    
    if keyword not in text_lower:
        return None
    
    pos = text_lower.find(keyword)
    after = text[pos + len(keyword):]
    words = after.split()
    
    return ' '.join(words[:3]).rstrip('.,;:')



### Задание 3.4 — Обогащение данных

Для каждого **валидного** сообщения добавьте поля:
- `amounts` — список извлечённых сумм (функция `extract_amounts`)
- `has_court_mention` — `True`/`False` (функция `find_court_mentions`)
- `manager_name` — ФИО или `None` (функция `extract_manager_name`)

In [115]:
for msg in valid_messages:
    msg['amounts'] = extract_amounts(msg['msg_text'])
    msg['has_court_mention'] = find_court_mentions(msg['msg_text'])
    msg['manager_name'] = extract_manager_name(msg['msg_text'])



### Задание 3.5 — Аналитика

Постройте следующие показатели по **валидным** сообщениям:

1. **Количество сообщений по типам** — словарь `{тип: количество}`
2. **Количество сообщений по регионам** — словарь `{регион: количество}`, пропуская `None`
3. **Топ-5 публикаторов** по количеству сообщений — `sorted(key=lambda...)`
4. **Таймлайн** — сообщения, отсортированные по дате публикации

In [117]:
messages_by_type = {}
for msg in valid_messages:
    msg_type = msg['type']
    messages_by_type[msg_type] = messages_by_type.get(msg_type, 0) + 1

print('1. Количество сообщений по типам:')
for msg_type, count in messages_by_type.items():
    print(f'  {msg_type}: {count}')

messages_by_region = {}
for msg in valid_messages:
    region = msg.get('region')
    if region is None:
        continue
    messages_by_region[region] = messages_by_region.get(region, 0) + 1

print('\n2. Количество сообщений по регионам:')
for region, count in messages_by_region.items():
    print(f'  {region}: {count}')

publishers = {}
for msg in valid_messages:
    inn = msg['publisher_inn']
    publishers[inn] = publishers.get(inn, 0) + 1

top5 = sorted(publishers.items(), key=lambda x: x[1], reverse=True)[:5]

print('\n3. Топ-5 публикаторов:')
for inn, count in top5:
    print(f'  ИНН {inn}: {count} сообщений')

timeline = sorted(valid_messages, key=lambda x: x['date_published'])

print('\n4. Таймлайн:')
for msg in timeline:
    print(f"  {msg['date_published'].strftime('%d.%m.%Y')} | {msg['publisher_inn']} | {msg['case_number']} | {msg['type']}")

1. Количество сообщений по типам:
  Введение процедуры: 8
  Собрание кредиторов: 3
  Завершение процедуры: 6
  Признание банкротом: 3
  Продажа имущества: 6
  Требование кредитора: 4
  Оспаривание сделки: 3
  Подача заявления: 3
  Отстранение управляющего: 1
  Мировое соглашение: 3
  Субсидиарная ответственность: 2
  Передача дела: 3
  Назначение управляющего: 2
  Жалоба на управляющего: 1

2. Количество сообщений по регионам:
  Москва: 25
  Свердловская область: 6
  Санкт-Петербург: 5
  Краснодарский край: 3
  Челябинская область: 3
  Ростовская область: 2
  Владимирская область: 1
  Московская область: 2

3. Топ-5 публикаторов:
  ИНН 7701234567: 3 сообщений
  ИНН 7706567890: 3 сообщений
  ИНН 7702345678: 2 сообщений
  ИНН 6658123450: 2 сообщений
  ИНН 7810345612: 2 сообщений

4. Таймлайн:
  10.01.2025 | 6658123450 | А60-25814/2025 | Завершение процедуры
  15.01.2025 | 7701234567 | А60-12345/2025 | Введение процедуры
  15.01.2025 | 5027123456 | А60-88888/2025 | Подача заявления
  18.0

---
# 🟣 Отчётность (0-1 балл)
---

### Задание 4.1 — Подготовка данных для сохранения

Подготовьте данные для записи в JSON. Даты нужно преобразовать обратно в строки (`strftime`),
чтобы JSON был читаемым.

In [118]:
export_messages = []

for msg in valid_messages:
    export_msg = {
        'publisher_inn': msg['publisher_inn'],
        'msg_text': msg['msg_text'],
        'date_published': msg['date_published'].strftime('%Y-%m-%d'),
        'type': msg['type'],
        'case_number': msg['case_number'],
        'region': msg['region'],
        'org_name': msg['org_name'],
        'amounts': msg['amounts'],
        'has_court_mention': msg['has_court_mention'],
        'manager_name': msg['manager_name'],
    }
    export_messages.append(export_msg)

# # Проверка
# print(f'Подготовлено записей: {len(export_messages)}')
# print(export_messages[0])

### Задание 4.2 — Сохранение `analysis_results.json`

Сохраните файл `analysis_results.json` со следующей структурой:
```json
{
  "valid_messages": [...],
  "type_stats": {...},
  "region_stats": {...},
  "priority_messages": [...]
}
```

In [119]:
priority_messages = [
    msg for msg in export_messages
    if msg['case_number'] in common_cases
]


analysis_results = {
    'valid_messages': export_messages,
    'type_stats': messages_by_type,
    'region_stats': messages_by_region,
    'priority_messages': priority_messages,
}

with open('analysis_results.json', 'w', encoding='utf-8') as f:
    json.dump(analysis_results, f, ensure_ascii=False, indent=2)

print(f'Файл сохранён.')
print(f'Валидных сообщений: {len(export_messages)}')
print(f'Приоритетных сообщений: {len(priority_messages)}')


Файл сохранён.
Валидных сообщений: 48
Приоритетных сообщений: 32


### Задание 4.3 — Сохранение `validation_errors.json`

Сохраните проблемные записи с описанием ошибок.

In [120]:
with open('validation_errors.json', 'w', encoding='utf-8') as f:
    json.dump(error_records, f, ensure_ascii=False, indent=2)

print(f'Файл сохранён.')
print(f'Записей с ошибками: {len(error_records)}')

Файл сохранён.
Записей с ошибками: 6


### Задание 4.4 — Текстовый отчёт `summary_report.txt`

Сформируйте текстовый аналитический отчёт для руководства с помощью **f-строк**.

Отчёт должен содержать:
- Заголовок и дату
- Общую статистику (всего сообщений, валидных, ошибочных)
- Статистику по типам сообщений
- Статистику по регионам
- Топ-5 публикаторов
- Список приоритетных дел
- Список найденных арбитражных управляющих

In [122]:
report = f"""
{'=' * 60}
АНАЛИТИЧЕСКИЙ ОТЧЁТ ПО ДАННЫМ ЕФРСБ
Дата формирования: {datetime.now().strftime('%d.%m.%Y')}
{'=' * 60}

1. ОБЩАЯ СТАТИСТИКА
   Всего сообщений:    {len(messages):>5}
   Валидных:           {len(valid_messages):>5}
   С ошибками:         {len(error_records):>5}

2. СТАТИСТИКА ПО ТИПАМ СООБЩЕНИЙ
{''.join(f'   {t}: {c}\n' for t, c in messages_by_type.items())}
3. СТАТИСТИКА ПО РЕГИОНАМ
{''.join(f'   {r}: {c}\n' for r, c in messages_by_region.items())}
4. ТОП-5 ПУБЛИКАТОРОВ
{''.join(f'   ИНН {inn}: {c} сообщений\n' for inn, c in top5)}
5. ПРИОРИТЕТНЫЕ ДЕЛА ({len(common_cases)} сообщений)
{''.join(f'   {c}\n' for c in sorted(common_cases))}
6. АРБИТРАЖНЫЕ УПРАВЛЯЮЩИЕ
{''.join(
    f"   {msg['case_number']} — {msg['manager_name']}\n"
    for msg in valid_messages
    if msg['manager_name'] is not None
)}
{'=' * 60}
"""

with open('summary_report.txt', 'w', encoding='utf-8') as f:
    f.write(report)

print(report)


АНАЛИТИЧЕСКИЙ ОТЧЁТ ПО ДАННЫМ ЕФРСБ
Дата формирования: 19.04.2026

1. ОБЩАЯ СТАТИСТИКА
   Всего сообщений:       54
   Валидных:              48
   С ошибками:             6

2. СТАТИСТИКА ПО ТИПАМ СООБЩЕНИЙ
   Введение процедуры: 8
   Собрание кредиторов: 3
   Завершение процедуры: 6
   Признание банкротом: 3
   Продажа имущества: 6
   Требование кредитора: 4
   Оспаривание сделки: 3
   Подача заявления: 3
   Отстранение управляющего: 1
   Мировое соглашение: 3
   Субсидиарная ответственность: 2
   Передача дела: 3
   Назначение управляющего: 2
   Жалоба на управляющего: 1

3. СТАТИСТИКА ПО РЕГИОНАМ
   Москва: 25
   Свердловская область: 6
   Санкт-Петербург: 5
   Краснодарский край: 3
   Челябинская область: 3
   Ростовская область: 2
   Владимирская область: 1
   Московская область: 2

4. ТОП-5 ПУБЛИКАТОРОВ
   ИНН 7701234567: 3 сообщений
   ИНН 7706567890: 3 сообщений
   ИНН 7702345678: 2 сообщений
   ИНН 6658123450: 2 сообщений
   ИНН 7810345612: 2 сообщений

5. ПРИОРИТЕТНЫЕ ДЕЛА 

---
## ✅ Итоги

Если вы корректно выполнили все 4 уровня, у вас должны получиться:

| Файл | Описание |
|------|----------|
| `analysis_results.json` | Валидные сообщения + статистика + приоритетные дела |
| `validation_errors.json` | Проблемные записи с описанием ошибок |
| `summary_report.txt` | Текстовый аналитический отчёт для руководства |

